# Simons Technical Scanner — BIST

**Strateji Özeti:**  
Geçmişte büyük yükseliş yapmış hisselerin fiyat hareketlerini analiz ederek,  
bu hareketlere benzer kurulumda olan güncel hisseleri tespit eder.

**Filtre Katmanları:**
1. **RSC (Relative Strength Comparison)** — Endekse göre göreceli güç
2. **MGB 4S** — Momentum + hacim kırılım sinyali
3. **Mira 4S** — Yükseliş öncesi sessiz birikim tespiti
4. **Yatay Destek/Direnç** — Konsolidasyon kırılımı
5. **Orta Vadeli İndikatör** — Trend yönü doğrulama
6. **RUA** — Hacim ağırlıklı momentum onayı


In [ ]:
# ── Kurulum ──────────────────────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg_spec, label=None):
    label = label or pkg_spec
    print(f"  Kuruluyor: {label} ...", end=" ")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg_spec],
        capture_output=True, text=True
    )
    print("OK" if result.returncode == 0 else f"HATA\n{result.stderr[-400:]}")

# Temel paketler
for pkg in [
    "pandas",
    "numpy",
    "requests",
    "ta",                      # Teknik analiz indikatörleri
    "tqdm",                    # Progress bar
    "openpyxl",                # Excel export
    "websocket-client",        # tvdatafeed bağımlılığı
    "websockets",              # tvdatafeed bağımlılığı
    "tradingview-screener",    # BIST hisse listesi
]:
    pip_install(pkg)

# tvdatafeed — GitHub (PyPI versiyonu eski)
pip_install(
    "git+https://github.com/rongardF/tvdatafeed.git",
    label="tvdatafeed (GitHub)"
)

# Google Colab Drive bağlantısı
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/Simons_Scanner"
    IN_COLAB = True
except Exception:
    DRIVE_ROOT = "/tmp/Simons_Scanner"
    IN_COLAB = False

import os
os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/raporlar", exist_ok=True)
print(f"\nDrive root: {DRIVE_ROOT}  |  Colab: {IN_COLAB}")


In [ ]:
# ── Import'lar ────────────────────────────────────────────────────────────────
import warnings, time, pickle
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Konfigürasyon ─────────────────────────────────────────────────────────────

# Tarama parametreleri
LOOKBACK_DAYS        = 252   # Geçmiş yükseliş analizi için gün sayısı (1 yıl)
CONSOLIDATION_DAYS   = 20    # Yatay konsolidasyon penceresi
MIN_HIST_RISE        = 0.40  # Geçmişte sayılan "büyük yükseliş" eşiği (%40)
MIN_VOLUME_RATIO     = 1.5   # Kırılım günü hacim / ortalama hacim oranı
RSC_PERIOD           = 50    # Göreceli güç periyodu (gün)
MGB_FAST             = 5     # MGB hızlı EMA
MGB_SLOW             = 20    # MGB yavaş EMA
MGB_SIGNAL           = 3     # MGB sinyal EMA
MIRA_LOOKBACK        = 20    # Mira 4S birikim penceresi
OVI_PERIOD           = 50    # Orta vadeli indikatör MA periyodu
RUA_PERIOD           = 14    # RUA hacim momentum periyodu
MIN_SCORE            = 4     # Minimum toplam puan (6 üzerinden)
CACHE_TTL_HOURS      = 6     # Cache geçerlilik süresi

print("Konfigürasyon yüklendi.")


In [ ]:
# ── BIST Hisse Listesi ────────────────────────────────────────────────────────
from tradingview_screener import Query, col

def get_bist_symbols(min_market_cap_m=None):
    """
    TradingView Screener ile BIST hisselerini çeker.
    min_market_cap_m: Minimum piyasa değeri (milyon TL), None = filtre yok
    """
    q = (
        Query()
        .select("name", "close", "volume", "market_cap_basic", "sector")
        .where(
            col("exchange").isin(["BIST"]),
            col("type") == "stock",
        )
        .limit(500)
    )
    if min_market_cap_m:
        q = q.where(col("market_cap_basic") > min_market_cap_m * 1e6)

    try:
        count, df = q.get_scanner_data()
        symbols = df["name"].str.replace("BIST:", "").tolist()
        print(f"BIST hisse sayısı: {len(symbols)}")
        return symbols, df
    except Exception as e:
        print(f"Screener hatası: {e}")
        # Fallback: BIST-100 sabit listesi
        fallback = [
            "AKBNK","AKSEN","ALARK","ARCLK","ASELS","BIMAS","CIMSA",
            "DOHOL","EKGYO","ENKAI","EREGL","FROTO","GARAN","GUBRF",
            "HALKB","ISCTR","KCHOL","KOZAA","KOZAL","KRDMD","MGROS",
            "ODAS","OTKAR","OYAKC","PETKM","PGSUS","SAHOL","SASA",
            "SISE","TAVHL","TCELL","THYAO","TKFEN","TOASO","TSKB",
            "TTKOM","TTRAK","TUPRS","VAKBN","VESTL","YKBNK",
        ]
        return fallback, None

SYMBOLS, SCREENER_DF = get_bist_symbols(min_market_cap_m=500)
print(f"Taranacak hisse sayısı: {len(SYMBOLS)}")


In [ ]:
# ── Fiyat Verisi Çekme ────────────────────────────────────────────────────────
from tvDatafeed import TvDatafeed, Interval

tv = TvDatafeed()  # Anonim bağlantı

def _cache_file(ticker: str) -> Path:
    return Path(DRIVE_ROOT) / "cache" / f"price_{ticker}.pkl"

def get_price_data(ticker: str, n_bars: int = 300) -> pd.DataFrame:
    """
    Günlük OHLCV verisi çeker. Cache destekli.
    Dönüş: datetime index'li DataFrame (open, high, low, close, volume)
    """
    cf = _cache_file(ticker)
    if cf.exists():
        age_h = (time.time() - cf.stat().st_mtime) / 3600
        if age_h < CACHE_TTL_HOURS:
            with open(cf, "rb") as f:
                return pickle.load(f)

    for attempt in range(3):
        try:
            df = tv.get_hist(
                symbol=ticker,
                exchange="BIST",
                interval=Interval.in_daily,
                n_bars=n_bars,
            )
            if df is not None and len(df) > 30:
                df.index = pd.to_datetime(df.index)
                df.columns = [c.lower() for c in df.columns]
                with open(cf, "wb") as f:
                    pickle.dump(df, f)
                return df
        except Exception:
            time.sleep(2 ** attempt)
    return pd.DataFrame()


In [ ]:
# ── İndikatör Hesaplama Fonksiyonları ────────────────────────────────────────

def calc_rsc(close: pd.Series, benchmark_close: pd.Series, period: int = RSC_PERIOD) -> float:
    """
    RSC (Relative Strength Comparison):
    Hissenin son N günlük getirisi / benchmark getirisi.
    > 1.0 → endeksten güçlü
    """
    if len(close) < period or len(benchmark_close) < period:
        return np.nan
    stock_ret = close.iloc[-1] / close.iloc[-period] - 1
    bench_ret = benchmark_close.iloc[-1] / benchmark_close.iloc[-period] - 1
    if bench_ret == 0:
        return np.nan
    return (1 + stock_ret) / (1 + bench_ret)


def calc_mgb_4s(close: pd.Series, volume: pd.Series) -> dict:
    """
    MGB 4S — Momentum + Hacim Kırılım Sinyali:
    MACD benzeri momentum hesabı, hacim filtresiyle güçlendirilmiş.
    4 sinyal kriteri (4S):
      1. Hızlı EMA > Yavaş EMA (fiyat momentum)
      2. MACD > Sinyal (momentum ivmesi)
      3. Hacim artışı (son bar / N-bar ortalama > eşik)
      4. Fiyat 20 günlük yüksek üzerinde
    """
    if len(close) < MGB_SLOW + 10:
        return {"signal": False, "score": 0, "macd": np.nan}

    ema_fast = close.ewm(span=MGB_FAST, adjust=False).mean()
    ema_slow = close.ewm(span=MGB_SLOW, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=MGB_SIGNAL, adjust=False).mean()

    vol_avg = volume.rolling(20).mean()
    vol_ratio = volume.iloc[-1] / vol_avg.iloc[-1] if vol_avg.iloc[-1] > 0 else 0

    high_20 = close.rolling(20).max().iloc[-2]  # Önceki günün 20-bar yüksek

    s1 = ema_fast.iloc[-1] > ema_slow.iloc[-1]
    s2 = macd_line.iloc[-1] > signal_line.iloc[-1]
    s3 = vol_ratio >= MIN_VOLUME_RATIO
    s4 = close.iloc[-1] > high_20

    score = sum([s1, s2, s3, s4])
    return {
        "signal": score >= 3,
        "score": score,
        "macd": float(macd_line.iloc[-1]),
        "vol_ratio": float(vol_ratio),
    }


def calc_mira_4s(close: pd.Series, volume: pd.Series) -> dict:
    """
    Mira 4S — Yükseliş Öncesi Sessiz Birikim Tespiti:
    Düşük volatilite + artan hacim = akıllı para girişi.
    4 kriter:
      1. Fiyat volatilitesi son N günde düşük (dar bant)
      2. Hacim son N günde artış trendinde
      3. Kapanış, N-günlük bandın üst %25'inde
      4. Düşük hacim günlerde fiyat düşüşü, yüksek hacim günlerde artış
    """
    n = MIRA_LOOKBACK
    if len(close) < n + 5:
        return {"signal": False, "score": 0}

    recent_close = close.iloc[-n:]
    recent_vol   = volume.iloc[-n:]

    price_range  = (recent_close.max() - recent_close.min()) / recent_close.mean()
    vol_slope    = np.polyfit(range(n), recent_vol.values, 1)[0]

    band_pos = (close.iloc[-1] - recent_close.min()) / (recent_close.max() - recent_close.min() + 1e-9)

    # Hacim/fiyat uyumu: yüksek hacimli günlerde fiyat artışı
    vol_median = recent_vol.median()
    high_vol_days  = recent_close[recent_vol > vol_median]
    low_vol_days   = recent_close[recent_vol <= vol_median]
    vol_price_ok   = (
        high_vol_days.pct_change().mean() > low_vol_days.pct_change().mean()
        if len(high_vol_days) > 2 and len(low_vol_days) > 2
        else False
    )

    m1 = price_range < 0.15          # Dar bant (<%15)
    m2 = vol_slope > 0               # Hacim artış trendi
    m3 = band_pos > 0.75             # Bandın üst kısmında
    m4 = vol_price_ok                # Hacim/fiyat uyumu

    score = sum([m1, m2, m3, m4])
    return {
        "signal": score >= 3,
        "score": score,
        "price_range_pct": round(price_range * 100, 2),
        "band_position": round(band_pos, 2),
    }


def calc_horizontal_sr(close: pd.Series, high: pd.Series, low: pd.Series) -> dict:
    """
    Yatay Destek/Direnç Kırılımı:
    Son N günde pivot yüksek/düşük seviyeleri tespit eder.
    Fiyat bu seviyeleri kırarsa sinyal üretir.
    """
    n = CONSOLIDATION_DAYS
    if len(close) < n + 10:
        return {"signal": False, "near_resistance": False}

    # Konsolidasyon penceresi (son N gün hariç 5 gün öncesi)
    window_close = close.iloc[-(n+5):-5]
    resistance   = window_close.quantile(0.90)
    support      = window_close.quantile(0.10)

    current = close.iloc[-1]
    prev    = close.iloc[-2]

    # Direnç kırılımı: fiyat direnç üstüne geçmiş
    breakout = prev <= resistance and current > resistance

    # Dirençe yakın ama henüz kırmamış (hazırlık)
    near_resistance = (resistance - current) / resistance < 0.03 and current <= resistance

    return {
        "signal": breakout,
        "near_resistance": near_resistance,
        "resistance": round(resistance, 2),
        "support": round(support, 2),
        "breakout_pct": round((current / resistance - 1) * 100, 2),
    }


def calc_ovi(close: pd.Series) -> dict:
    """
    Orta Vadeli İndikatör (OVI):
    Fiyatın orta vadeli trend üzerinde olup olmadığını ölçer.
    Hızlı MA > Yavaş MA → yükselen trend
    """
    if len(close) < OVI_PERIOD + 10:
        return {"signal": False, "trend": "belirsiz"}

    ma_fast = close.rolling(OVI_PERIOD // 2).mean()
    ma_slow = close.rolling(OVI_PERIOD).mean()
    ma_200  = close.rolling(200).mean() if len(close) >= 200 else None

    above_fast = close.iloc[-1] > ma_fast.iloc[-1]
    ma_cross   = ma_fast.iloc[-1] > ma_slow.iloc[-1]
    above_200  = (close.iloc[-1] > ma_200.iloc[-1]) if ma_200 is not None and not pd.isna(ma_200.iloc[-1]) else None

    signal = above_fast and ma_cross
    trend  = "yükselen" if signal else "düşen"
    if above_200 is True:
        trend += " (200MA üstü)"

    return {
        "signal": signal,
        "trend": trend,
        "ma_fast": round(ma_fast.iloc[-1], 2),
        "ma_slow": round(ma_slow.iloc[-1], 2),
    }


def calc_rua(close: pd.Series, volume: pd.Series) -> dict:
    """
    RUA — Hacim Ağırlıklı Momentum Onayı:
    OBV (On-Balance Volume) trendine dayalı.
    OBV yükselen trend + RSI momentum onayı.
    """
    if len(close) < RUA_PERIOD + 5:
        return {"signal": False, "obv_trend": 0, "rsi": np.nan}

    # OBV hesapla
    direction = np.sign(close.diff())
    obv = (direction * volume).cumsum()

    # OBV eğimi (son 14 gün)
    obv_slope = np.polyfit(range(RUA_PERIOD), obv.iloc[-RUA_PERIOD:].values, 1)[0]

    # RSI
    delta  = close.diff()
    gain   = delta.where(delta > 0, 0).rolling(14).mean()
    loss   = (-delta).where(delta < 0, 0).rolling(14).mean()
    rs     = gain / (loss + 1e-9)
    rsi    = 100 - (100 / (1 + rs))

    rsi_val = float(rsi.iloc[-1])
    # RSI: 50-70 arası ideal momentum bölgesi
    rsi_ok  = 45 <= rsi_val <= 75

    signal = obv_slope > 0 and rsi_ok
    return {
        "signal": signal,
        "obv_trend": round(obv_slope, 0),
        "rsi": round(rsi_val, 1),
    }


print("İndikatör fonksiyonları tanımlandı.")


In [ ]:
# ── Geçmiş Yükseliş Analizi ───────────────────────────────────────────────────

def analyze_historical_rises(df: pd.DataFrame, min_rise: float = MIN_HIST_RISE) -> list:
    """
    Geçmişte büyük yükseliş yapan dönemleri tespit eder.
    Her yükseliş için: başlangıç fiyatı, zirve fiyatı, süre, önceki konsolidasyon.
    """
    if len(df) < 60:
        return []

    close = df["close"]
    rises = []

    # Rolling window ile büyük yükselişleri bul
    for window in [20, 40, 60]:
        roll_max = close.rolling(window).max()
        roll_min = close.rolling(window).min().shift(window // 2)
        rise_pct = (roll_max - roll_min) / (roll_min + 1e-9)

        big_rise_idx = rise_pct[rise_pct >= min_rise].index
        if len(big_rise_idx) > 0:
            for idx in big_rise_idx[-5:]:  # Son 5 büyük yükseliş
                rises.append({
                    "date": idx,
                    "rise_pct": round(float(rise_pct[idx]) * 100, 1),
                    "window_days": window,
                })

    return rises


def extract_pre_rise_pattern(df: pd.DataFrame) -> dict:
    """
    Büyük yükseliş ÖNCESI 20 günün karakteristik özelliklerini çıkarır.
    Bu pattern, güncel hisselerde aranacak kurulum.
    """
    if len(df) < 60:
        return {}

    close  = df["close"]
    volume = df["volume"]

    # Tüm periyotta en büyük yükselişin başlangıç noktasını bul
    max_idx  = close.argmax()
    if max_idx < 20:
        return {}

    pre_rise = close.iloc[max_idx - 20: max_idx]
    pre_vol  = volume.iloc[max_idx - 20: max_idx]

    return {
        "pre_rise_volatility": float(pre_rise.std() / pre_rise.mean()),
        "pre_rise_vol_trend":  float(np.polyfit(range(20), pre_vol.values, 1)[0]),
        "pre_rise_band_pct":   float((pre_rise.max() - pre_rise.min()) / pre_rise.mean()),
    }


print("Geçmiş yükseliş analiz fonksiyonları hazır.")


In [ ]:
# ── Benchmark (BIST-100) Verisi ───────────────────────────────────────────────

print("BIST-100 benchmark verisi çekiliyor...")
BIST100_DF = get_price_data("XU100", n_bars=400)

if BIST100_DF.empty:
    # tvdatafeed ile XU100 çekilemediyse TRY ile dene
    for sym in ["BIST100", "XU100", "XU030"]:
        BIST100_DF = get_price_data(sym, n_bars=400)
        if not BIST100_DF.empty:
            break

if not BIST100_DF.empty:
    print(f"BIST-100 verisi hazır. Satır: {len(BIST100_DF)}")
    BENCHMARK_CLOSE = BIST100_DF["close"]
else:
    print("UYARI: Benchmark verisi alınamadı, RSC hesaplanamayacak.")
    BENCHMARK_CLOSE = None


In [ ]:
# ── Ana Tarama Fonksiyonu ────────────────────────────────────────────────────

def scan_single_stock(ticker: str) -> dict | None:
    """
    Tek hisse için tüm Simons indikatörlerini hesaplar.
    MIN_SCORE koşulunu sağlamazsa None döner.
    """
    df = get_price_data(ticker, n_bars=350)
    if df.empty or len(df) < 60:
        return None

    close  = df["close"]
    high   = df["high"]
    low    = df["low"]
    volume = df["volume"]

    # ── 1. RSC ────────────────────────────────────────────────────────────────
    rsc_val = 0
    if BENCHMARK_CLOSE is not None:
        # Benchmark ile hisseyi aynı uzunluğa getir
        common = close.index.intersection(BENCHMARK_CLOSE.index)
        if len(common) >= RSC_PERIOD:
            rsc_val = calc_rsc(
                close.loc[common].iloc[-RSC_PERIOD:],
                BENCHMARK_CLOSE.loc[common].iloc[-RSC_PERIOD:],
            )
    rsc_signal = isinstance(rsc_val, float) and rsc_val > 1.05

    # ── 2. MGB 4S ─────────────────────────────────────────────────────────────
    mgb = calc_mgb_4s(close, volume)

    # ── 3. Mira 4S ────────────────────────────────────────────────────────────
    mira = calc_mira_4s(close, volume)

    # ── 4. Yatay Destek/Direnç ────────────────────────────────────────────────
    sr = calc_horizontal_sr(close, high, low)

    # ── 5. Orta Vadeli İndikatör ──────────────────────────────────────────────
    ovi = calc_ovi(close)

    # ── 6. RUA ────────────────────────────────────────────────────────────────
    rua = calc_rua(close, volume)

    # ── Geçmiş Yükseliş Analizi ───────────────────────────────────────────────
    hist_rises  = analyze_historical_rises(df)
    pre_pattern = extract_pre_rise_pattern(df)

    # ── Toplam Skor ───────────────────────────────────────────────────────────
    signals = [
        bool(rsc_signal),
        bool(mgb.get("signal")),
        bool(mira.get("signal")),
        bool(sr.get("signal") or sr.get("near_resistance")),
        bool(ovi.get("signal")),
        bool(rua.get("signal")),
    ]
    total_score = sum(signals)

    if total_score < MIN_SCORE:
        return None

    return {
        "ticker":           ticker,
        "fiyat":            round(float(close.iloc[-1]), 2),
        "toplam_skor":      total_score,
        # RSC
        "RSC":              round(rsc_val, 3) if isinstance(rsc_val, float) else "-",
        "RSC_sinyal":       "✓" if rsc_signal else "✗",
        # MGB 4S
        "MGB_skor":         mgb.get("score", 0),
        "MGB_sinyal":       "✓" if mgb.get("signal") else "✗",
        "hacim_oran":       round(mgb.get("vol_ratio", 0), 2),
        # Mira 4S
        "Mira_skor":        mira.get("score", 0),
        "Mira_sinyal":      "✓" if mira.get("signal") else "✗",
        "bant_pct":         mira.get("price_range_pct", "-"),
        # Yatay SR
        "SR_kirilim":       "✓" if sr.get("signal") else ("~" if sr.get("near_resistance") else "✗"),
        "direnc":           sr.get("resistance", "-"),
        # OVI
        "OVI_sinyal":       "✓" if ovi.get("signal") else "✗",
        "trend":            ovi.get("trend", "-"),
        # RUA
        "RUA_sinyal":       "✓" if rua.get("signal") else "✗",
        "RSI":              rua.get("rsi", "-"),
        # Geçmiş
        "gecmis_yukselis":  len(hist_rises),
        "max_yukselis_pct": max([r["rise_pct"] for r in hist_rises], default=0),
    }


print("Tarama fonksiyonu hazır.")


In [ ]:
# ── Tüm BIST Taraması ─────────────────────────────────────────────────────────

print(f"Tarama başlıyor... {len(SYMBOLS)} hisse")
print(f"Minimum skor eşiği: {MIN_SCORE}/6")
print("-" * 50)

results = []
errors  = []

for ticker in tqdm(SYMBOLS, desc="Taranıyor"):
    try:
        result = scan_single_stock(ticker)
        if result:
            results.append(result)
    except Exception as e:
        errors.append((ticker, str(e)))
    time.sleep(0.1)  # API rate limit

print(f"\nTarama tamamlandı.")
print(f"Sinyal veren hisse: {len(results)}")
print(f"Hata: {len(errors)}")


In [ ]:
# ── Sonuçları Göster ─────────────────────────────────────────────────────────

if not results:
    print("Sinyal veren hisse bulunamadı. MIN_SCORE eşiğini düşürmeyi dene.")
else:
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(
        ["toplam_skor", "max_yukselis_pct"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Öncelik sütunları
    display_cols = [
        "ticker", "fiyat", "toplam_skor",
        "RSC", "RSC_sinyal",
        "MGB_skor", "MGB_sinyal", "hacim_oran",
        "Mira_skor", "Mira_sinyal", "bant_pct",
        "SR_kirilim", "direnc",
        "OVI_sinyal", "trend",
        "RUA_sinyal", "RSI",
        "gecmis_yukselis", "max_yukselis_pct",
    ]

    print(f"\n{'='*70}")
    print(f"SIMONS TECHNICAL SCANNER — {datetime.now().strftime('%d.%m.%Y %H:%M')}")
    print(f"{'='*70}")
    print(f"Toplam sinyal: {len(df_results)} hisse")
    print()

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_rows", 50)

    print(df_results[display_cols].to_string(index=True))

    # ── Excel çıktısı ─────────────────────────────────────────────────────────
    excel_path = f"{DRIVE_ROOT}/raporlar/simons_tarama_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
    df_results.to_excel(excel_path, index=False)
    print(f"\nExcel kaydedildi: {excel_path}")


In [ ]:
# ── Hisse Detay Analizi ───────────────────────────────────────────────────────
# Tek bir hisseyi detaylı incele

def detailed_analysis(ticker: str):
    """
    Seçili hisse için tam Simons analiz raporu.
    """
    print(f"\n{'='*60}")
    print(f"  {ticker} — SİMONS TEKNİK ANALİZ")
    print(f"{'='*60}")

    df = get_price_data(ticker, n_bars=350)
    if df.empty:
        print("Veri alınamadı.")
        return

    close  = df["close"]
    volume = df["volume"]
    high   = df["high"]
    low    = df["low"]

    print(f"\nFiyat    : {close.iloc[-1]:.2f} TL")
    print(f"1G değişim: {(close.iloc[-1]/close.iloc[-2]-1)*100:.2f}%")
    print(f"1H değişim: {(close.iloc[-1]/close.iloc[-5]-1)*100:.2f}%")
    print(f"1A değişim: {(close.iloc[-1]/close.iloc[-21]-1)*100:.2f}%")

    print(f"\n--- RSC (Endekse Göre Güç) ---")
    if BENCHMARK_CLOSE is not None:
        common = close.index.intersection(BENCHMARK_CLOSE.index)
        rsc = calc_rsc(close.loc[common], BENCHMARK_CLOSE.loc[common])
        print(f"  RSC (50G): {rsc:.3f}  {'✓ Endeksten GÜÇLÜ' if rsc > 1.05 else '✗ Endeksten zayıf'}")

    print(f"\n--- MGB 4S (Momentum + Hacim) ---")
    mgb = calc_mgb_4s(close, volume)
    print(f"  Skor     : {mgb['score']}/4")
    print(f"  Hacim    : {mgb['vol_ratio']:.2f}x ortalama")
    print(f"  MACD     : {mgb['macd']:.4f}")

    print(f"\n--- Mira 4S (Sessiz Birikim) ---")
    mira = calc_mira_4s(close, volume)
    print(f"  Skor     : {mira['score']}/4")
    print(f"  Bant %   : {mira['price_range_pct']}%")
    print(f"  Band Pos : {mira['band_position']} (>0.75 iyi)")

    print(f"\n--- Yatay Destek/Direnç ---")
    sr = calc_horizontal_sr(close, high, low)
    print(f"  Direnç   : {sr['resistance']} TL")
    print(f"  Destek   : {sr['support']} TL")
    print(f"  Kırılım  : {'✓ KIRILDI' if sr['signal'] else ('~ Yakın' if sr['near_resistance'] else '✗ Yok')}")

    print(f"\n--- Orta Vadeli İndikatör ---")
    ovi = calc_ovi(close)
    print(f"  Trend    : {ovi['trend']}")
    print(f"  MA Hızlı : {ovi['ma_fast']}")
    print(f"  MA Yavaş : {ovi['ma_slow']}")

    print(f"\n--- RUA (Hacim Momentum) ---")
    rua = calc_rua(close, volume)
    print(f"  RSI      : {rua['rsi']}")
    print(f"  OBV Trend: {'↑ Yükselen' if rua['obv_trend'] > 0 else '↓ Düşen'}")

    print(f"\n--- Geçmiş Yükselişler ---")
    rises = analyze_historical_rises(df)
    if rises:
        for r in sorted(rises, key=lambda x: x['rise_pct'], reverse=True)[:3]:
            print(f"  {r['date'].strftime('%Y-%m-%d')}: +{r['rise_pct']}% ({r['window_days']} günde)")
    else:
        print("  Büyük yükseliş tespit edilmedi.")

    print()


# Örnek kullanım — tarama sonuçlarında en yüksek skorlu hisseyi incele
if results:
    best_ticker = df_results.iloc[0]["ticker"]
    detailed_analysis(best_ticker)


In [ ]:
# ── İstediğin hisseyi detaylı incele ─────────────────────────────────────────
# Bu hücreyi değiştirerek istediğin hisseyi analiz edebilirsin.

INCELEME_HISSESI = "THYAO"   # <-- buraya istediğin ticker yaz

detailed_analysis(INCELEME_HISSESI)
